# DEQ Export — Rotated Surface Code

**[DEMO]** — `lightstim.deq` exports a rotated-surface-code memory experiment
to Microsoft's DEQ device DSL (`CODE`/`GADGET`/`COMPOSE`/`PROGRAM` blocks).
This notebook verifies the export three independent ways:

1. **Tier 0** — parse the emitted text with the real `deq` package and run its
   algebraic `CODE` validator (needs only `deq`, not the compiled runtime).
2. **Tier 1** — replay the exported gadget bodies back into a `stim.Circuit`
   and compare against LightStim's own native circuit construction.
3. **Tier 2** — transpile the exported `PROGRAM` with Microsoft's real
   compiler and sample it through DEQ's own runtime (needs `deq_runtime`);
   a correct, noiseless memory experiment must give an all-zero syndrome and
   an all-zero logical readout on every shot.

See `lightstim/deq/` for the exporter, `lightstim/deq/validate.py` for the
comparison helpers used below, and `tests/test_deq_export.py` /
`tests/test_run_deq_export.py` for the same checks as an automated suite.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "lightstim").is_dir() and (path / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import re
import importlib.util
import subprocess
import tempfile

from lightstim.deq.export import export_rotated_surface_code_memory
from lightstim.deq.gadgets import qubit_remap
from lightstim.deq.validate import (
    deq_available,
    gadget_body_to_stim_circuit,
    is_physical_instruction_line,
    parse_and_validate_codes,
    relabel_qubits,
    strip_annotations,
    strip_ticks,
)
from lightstim.ir.qec_system import QECSystem
from lightstim.protocols.memory import MemoryExperiment
from lightstim.qec_code.surface_code.rotated.code_patch import RotatedSurfaceCode

In [ ]:
distance = 3  # odd, >= 3; kept small for a readable demo
rounds = 3    # total syndrome-extraction rounds, matches MemoryExperiment(rounds=...)
basis = "Z"   # "Z" or "X"

## Export

In [ ]:
deq_text = export_rotated_surface_code_memory(distance=distance, rounds=rounds, basis=basis)
print(deq_text)

## Tier 0 — parse and validate with the real `deq` package

Microsoft's own parser plus its algebraic `CODE` validator (stabilizer
commutation, CSS structure). No `deq_runtime` needed for this tier.

In [ ]:
if deq_available():
    deq_file = parse_and_validate_codes(deq_text)
    kinds = [type(d).__name__ for d in deq_file.definitions]
    print("Parsed and validated OK. Definitions:", kinds)
else:
    print("`deq` is not installed (pip install -e '.[deq]'); skipping Tier 0.")

## Tier 1 — round-trip against LightStim's native circuit

Replay the exported `PrepareZ`/`SyndromeExtraction`/`MeasureZ` gadget bodies
back into a `stim.Circuit` and compare against
`MemoryExperiment(qec_patch=RotatedSurfaceCode(...)).build()` — the same
`assert circuit == native` pattern `tests/test_run_memory.py` uses for its
own CLI-vs-native cross-check.

Three normalizations are needed first, each a benign convention difference
rather than a correctness gap (see `lightstim/deq/validate.py`'s
docstrings): qubit renumbering (the exporter numbers data qubits 0..n-1 then
ancillas; LightStim's own global numbering interleaves them), TICK placement
at gadget-call boundaries (DEQ's compiler concatenates calls with no implicit
TICK; LightStim's native readout inserts one), and REPEAT-block flattening
(LightStim wraps steady-state rounds in a `stim.CircuitRepeatBlock` once
`rounds > 2`; the exporter inlines each call instead).

In [ ]:
def gadget_body(text, name):
    match = re.search(rf"GADGET {name} \{{\n(.*?)\n\}}", text, re.DOTALL)
    return [line.strip() for line in match.group(1).splitlines()]


prepare_name = "PrepareZ" if basis == "Z" else "PrepareX"
measure_name = "MeasureZ" if basis == "Z" else "MeasureX"

lines = [l for l in gadget_body(deq_text, prepare_name) if is_physical_instruction_line(l)]
se_lines = [l for l in gadget_body(deq_text, "SyndromeExtraction") if is_physical_instruction_line(l)]
for _ in range(rounds - 1):
    lines += se_lines
lines += [l for l in gadget_body(deq_text, measure_name) if is_physical_instruction_line(l)]
rebuilt = gadget_body_to_stim_circuit(lines)

native = MemoryExperiment(
    qec_patch=RotatedSurfaceCode(distance=distance), basis=basis, rounds=rounds, if_detector=True,
).build()
patch = QECSystem().add_patch(RotatedSurfaceCode(distance=distance), name="patch")
remap = qubit_remap(patch)
native_physical_only = relabel_qubits(strip_annotations(native), remap)

matches = strip_ticks(rebuilt).flattened() == strip_ticks(native_physical_only).flattened()
print(f"Rebuilt circuit matches LightStim's native circuit: {matches}")
assert matches

## Tier 2 — real DEQ compiler + runtime

Transpiles the exported `PROGRAM` with `python -m deq transpile`, then
samples it noiselessly with `python -m deq sample`. Needs the optional
`deq_runtime` package (`pip install -e '.[deq]'`); skipped gracefully if it
isn't installed.

In [ ]:
if importlib.util.find_spec("deq_runtime") is None:
    print("`deq_runtime` is not installed; skipping Tier 2. "
          "Install it (pip install -e '.[deq]') to run this cell for real.")
else:
    program_name = f"RotatedSurfaceCodeD{distance}MemoryExperiment{basis}{rounds}"
    work_dir = Path(tempfile.mkdtemp(prefix="lightstim_deq_demo_"))
    deq_path = work_dir / "device.deq"
    jit_path = work_dir / "device.deq.jit"
    deq_path.write_text(deq_text)

    subprocess.run(
        [sys.executable, "-m", "deq", "transpile", str(deq_path),
         "--program", program_name, "--out", str(jit_path)],
        check=True, capture_output=True,
    )
    result = subprocess.run(
        [sys.executable, "-m", "deq", "sample", str(deq_path), "--program", program_name,
         "--shots", "200", "--noiseless", "--interpret"],
        check=True, capture_output=True, text=True,
    )
    syndromes = [line for line in result.stdout.splitlines() if line.startswith("Syndrome:")]
    readouts = [line for line in result.stdout.splitlines() if line.startswith("Readout:")]
    nonzero_syndromes = sum(1 for line in syndromes if int(line.split()[-1], 2) != 0)
    nonzero_readouts = sum(1 for line in readouts if int(line.split()[-1], 2) != 0)
    print(f"{len(syndromes)} noiseless shots: {nonzero_syndromes} nonzero syndromes, {nonzero_readouts} nonzero readouts.")
    assert nonzero_syndromes == 0 and nonzero_readouts == 0

## Circuit diagram

LightStim's native circuit for a small, readable case (2 rounds) — the same
physical operations the export above encodes as `PrepareZ` +
`SyndromeExtraction` + `MeasureZ` gadgets.

In [ ]:
small = MemoryExperiment(qec_patch=RotatedSurfaceCode(distance=3), basis="Z", rounds=2).build()
small.without_noise().diagram("detslice-with-ops-svg")